In [ ]:
import CDutils
import scipy.io as sio
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import savemat
import os
from sklearn.metrics import f1_score

In [ ]:
path = r"D:\Code\Contraction_detection_manuscript_second_round\Contaction_detection\Data_sorted"
dir_list = os.listdir(path)

In [ ]:
dir_list

In [ ]:
f1_list = []
cp_score_list = []
cpn_list = []
for pt in range(len(dir_list)):
    data = sio.loadmat(path + '/' + dir_list[pt])
    modulation_ds = data['tmp_y']
    label_ds = data['newClusterSig']
    
    ch_num = modulation_ds.shape[0]
    ch_length = modulation_ds.shape[1]
    signal_length = modulation_ds.shape[1]
    
    modulation_ar_removal = CDutils.ar_remove_patient(modulation_ds)
    c_detection = np.zeros_like(modulation_ar_removal)

    zcr_array = np.zeros_like(modulation_ar_removal)
    zcr_modulation = np.zeros_like(modulation_ar_removal)
   
    
    seg_length = 100 #2.56Hz about 40s
    seg_num = ch_length-seg_length+1
    for ch in range(ch_num):
        sig_ch = modulation_ds[ch,:]
        for i in range(seg_num):
            sig_seg = sig_ch[i:i+seg_length]
            zcr_array[ch,i+49] = np.power(CDutils.zero_crossing_rate(sig_seg),3)
        zcr_array[ch,1:49] = zcr_array[ch,49]
        zcr_array[ch,50:] = zcr_array[ch,50]

    zcr_modulation = modulation_ar_removal*zcr_array  
    
    f1_array = np.zeros(ch_num,)
    cp_score_array = np.zeros(ch_num,)
    cpn_array = np.zeros(ch_num,)

    c_detection = np.zeros_like(modulation_ar_removal)
    window_length = 30
    for ch in range(ch_num):
        rms = CDutils.rms_envelope(zcr_modulation[ch,:],30)
        
        c_detection[ch,:] = CDutils.rms_cd(rms,window_length,0.4)
        pre_cpds = CDutils.find_cpd(c_detection[ch,:], 1)
        gt_cpds = CDutils.find_cpd(label_ds[ch,:], 1)
        f1_array[ch] = f1_score(label_ds[ch,:], c_detection[ch,:], zero_division=1.0)
        cp_score_array[ch] = CDutils.cp_score(gt_cpds,pre_cpds,signal_length)
        # cp_score_array1[ch] = CDutils.cp_score1(gt_cpds,pre_cpds,signal_length)
        cpn_array[ch] = (np.abs(len(gt_cpds)-len(pre_cpds)))/len(gt_cpds)
    print(np.median(f1_array))
    print(np.median(cp_score_array))
    print(np.median(cpn_array))
    f1_list.append(np.median(f1_array))
    cp_score_list.append(np.median(cp_score_array))
    cpn_list.append(np.median(cpn_array))

In [ ]:
mdic = {"f1_list": f1_list,"cp_score_list": cp_score_list,"cpn_list": cpn_list}
savemat(r"5_zcr.mat", mdic)